In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import joblib
prep = joblib.load('/content/drive/MyDrive/MLmodeling/XAI/splits/TON_IoT/preprocessor.pkl')
names = prep.get_feature_names_out()
print(f"Column 1: {names[1]}")
print(f"Feature 21: {names[21]}")
print(f"Total features: {len(names)}")

Column 1: dst_port
Column 21: ssl_cipher


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MLmodeling/XAI/splits/full_dataset/X_test.csv', nrows=5)
print(f"Total columns: {len(df.columns)}")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

Total columns: 411
  0: flow_duration
  1: header_length
  2: protocol_type
  3: duration
  4: rate
  5: srate
  6: drate
  7: fin_flag_number
  8: syn_flag_number
  9: rst_flag_number
  10: psh_flag_number
  11: ack_flag_number
  12: ack_count
  13: syn_count
  14: fin_count
  15: urg_count
  16: rst_count
  17: http
  18: https
  19: dns
  20: tcp
  21: udp
  22: arp
  23: icmp
  24: ipv
  25: llc
  26: tot_sum
  27: min
  28: max
  29: avg
  30: std
  31: tot_size
  32: iat
  33: number
  34: magnitue
  35: radius
  36: covariance
  37: variance
  38: weight
  39: id.orig_p
  40: id.resp_p
  41: fwd_pkts_tot
  42: bwd_pkts_tot
  43: fwd_data_pkts_tot
  44: bwd_data_pkts_tot
  45: fwd_pkts_per_sec
  46: bwd_pkts_per_sec
  47: flow_pkts_per_sec
  48: down_up_ratio
  49: fwd_header_size_tot
  50: fwd_header_size_min
  51: fwd_header_size_max
  52: bwd_header_size_tot
  53: bwd_header_size_min
  54: bwd_header_size_max
  55: flow_fin_flag_count
  56: flow_syn_flag_count
  57: flow_rst_f

In [ ]:
import os
import nbformat
from pathlib import Path
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.0 MB/s eta 0:00:00


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/My Drive/Praxis/Data'):
    for f in files:
        if 'test' in f.lower() and f.endswith('.csv'):
            print(os.path.join(root, f))

In [ ]:
"""
Bootstrap Confidence Intervals — All Models × All Datasets
Run in Google Colab with Drive mounted.
Outputs: bootstrap_results.csv
"""

import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score, precision_score, recall_score

# !pip install catboost  # ← uncomment if needed

BASE = '/content/drive/MyDrive/MLmodeling/XAI'

DATASETS = {
    'UNSW-NB15': {
        'split_dir': f'{BASE}/splits/UNSW-NB15',
        'model_dir': f'{BASE}/models/UNSW-NB15',
        'prefix': 'UNSW',
    },
    'RT-IoT2022': {
        'split_dir': f'{BASE}/splits/RT-IoT2022',
        'model_dir': f'{BASE}/models/RT-IoT2022',
        'prefix': 'RT',
    },
    'TON_IoT': {
        'split_dir': f'{BASE}/splits/TON_IoT',
        'model_dir': f'{BASE}/models/TON_IoT',
        'prefix': 'TON',
    },
    'CICIoT2023': {
        'split_dir': f'{BASE}/splits/CICIoT2023',
        'model_dir': f'{BASE}/models/CICIoT2023',
        'prefix': 'CIC',
    },
}

SKIP_MODELS = {'IsolationForest', 'LOF'}


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except:
            continue
    if not scores:
        return np.nan, np.nan, np.nan
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def load_test_data(config):
    X_test_df = pd.read_csv(f"{config['split_dir']}/X_test.csv")
    y_test = pd.read_csv(f"{config['split_dir']}/y_test.csv").squeeze()

    fn_path = f"{config['split_dir']}/feature_names.csv"
    feature_names = None
    if os.path.exists(fn_path):
        feature_names = pd.read_csv(fn_path, header=None).squeeze().tolist()

    # Check if data has string columns (not yet encoded)
    if X_test_df.select_dtypes(include=['object', 'category']).shape[1] > 0:
        # Load preprocessor and transform
        for ext in ['.pkl', '.joblib']:
            pp_path = f"{config['split_dir']}/preprocessor{ext}"
            if os.path.exists(pp_path):
                preprocessor = joblib.load(pp_path)
                X_test = preprocessor.transform(X_test_df)
                if hasattr(X_test, 'toarray'):
                    X_test = X_test.toarray()
                return np.asarray(X_test, dtype=float), np.asarray(y_test, dtype=int), feature_names
        raise ValueError("String columns found but no preprocessor available")
    else:
        return np.asarray(X_test_df, dtype=float), np.asarray(y_test, dtype=int), feature_names


all_results = []

for ds_name, config in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"  {ds_name}")
    print(f"{'='*60}")

    if not os.path.exists(f"{config['split_dir']}/X_test.csv"):
        print(f"  ⚠️  X_test.csv not found in {config['split_dir']}")
        continue

    try:
        X_test, y_test, feature_names = load_test_data(config)
        print(f"  Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
    except Exception as e:
        print(f"  ⚠️  Failed to load test data: {e}")
        continue

    # Find model files (skip Archive folder)
    model_files = []
    for f in sorted(os.listdir(config['model_dir'])):
        if f.endswith('.pkl') and f.startswith(config['prefix']):
            model_files.append(f)

    print(f"  Found {len(model_files)} models")

    for mf in model_files:
        model_path = os.path.join(config['model_dir'], mf)
        parts = mf.replace('.pkl', '').split('_')
        model_name = '_'.join(parts[2:]) if len(parts) >= 3 else mf.replace('.pkl', '')

        # Skip unsupervised models
        if model_name in SKIP_MODELS:
            print(f"\n  {model_name}... SKIPPED (unsupervised)")
            continue

        print(f"\n  {model_name}...", end=" ")

        try:
            model = joblib.load(model_path)

            expected_n = getattr(model, 'n_features_in_', X_test.shape[1])
            X_input = X_test

            if X_test.shape[1] != expected_n:
                if X_test.shape[1] > expected_n:
                    X_input = X_test[:, :expected_n]
                else:
                    print(f"SKIP (need {expected_n} features, have {X_test.shape[1]})")
                    continue

            y_pred = model.predict(X_input)

            f1 = f1_score(y_test, y_pred, zero_division=0)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)

            f1_mean, f1_lo, f1_hi = bootstrap_ci(y_test, y_pred, lambda yt, yp: f1_score(yt, yp, zero_division=0))
            prec_mean, prec_lo, prec_hi = bootstrap_ci(y_test, y_pred, lambda yt, yp: precision_score(yt, yp, zero_division=0))
            rec_mean, rec_lo, rec_hi = bootstrap_ci(y_test, y_pred, lambda yt, yp: recall_score(yt, yp, zero_division=0))

            print(f"F1={f1:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]")

            all_results.append({
                'dataset': ds_name,
                'model': model_name,
                'test_f1': f"{f1:.4f}",
                'f1_ci_lower': f"{f1_lo:.4f}",
                'f1_ci_upper': f"{f1_hi:.4f}",
                'test_precision': f"{prec:.4f}",
                'prec_ci_lower': f"{prec_lo:.4f}",
                'prec_ci_upper': f"{prec_hi:.4f}",
                'test_recall': f"{rec:.4f}",
                'rec_ci_lower': f"{rec_lo:.4f}",
                'rec_ci_upper': f"{rec_hi:.4f}",
            })

        except Exception as e:
            print(f"FAILED: {e}")
            all_results.append({
                'dataset': ds_name,
                'model': model_name,
                'test_f1': 'ERROR',
                'f1_ci_lower': str(e)[:50],
                'f1_ci_upper': '',
                'test_precision': '', 'prec_ci_lower': '', 'prec_ci_upper': '',
                'test_recall': '', 'rec_ci_lower': '', 'rec_ci_upper': '',
            })

results_df = pd.DataFrame(all_results)
out_path = f'{BASE}/bootstrap_results.csv'
results_df.to_csv(out_path, index=False)

print(f"\n\n{'='*60}")
print(f"  DONE — saved to {out_path}")
print(f"{'='*60}")
print(results_df.to_string(index=False))


  UNSW-NB15
  Test set: 38651 samples, 192 features
  Found 16 models

  LogisticRegression... F1=0.9286 [0.9263, 0.9309]

  DecisionTree... F1=0.9521 [0.9502, 0.9541]

  RandomForest... F1=0.9612 [0.9595, 0.9630]

  ExtraTrees... F1=0.9607 [0.9590, 0.9626]

  GradientBoosting... F1=0.9495 [0.9477, 0.9515]

  AdaBoost... F1=0.9315 [0.9292, 0.9337]

  XGBoost... F1=0.9591 [0.9575, 0.9608]

  LightGBM... F1=0.9575 [0.9556, 0.9595]

  CatBoost... FAILED: Input data must have at least one feature

  KNN... F1=0.9442 [0.9421, 0.9464]

  NaiveBayes... F1=0.3655 [0.3586, 0.3726]

  LinearSVM... F1=0.9291 [0.9268, 0.9314]

  MLP... F1=0.9537 [0.9519, 0.9557]

  BaggingClassifier... F1=0.9623 [0.9607, 0.9640]

  IsolationForest... SKIPPED (unsupervised)

  LOF... SKIPPED (unsupervised)

  RT-IoT2022
  Test set: 18468 samples, 93 features
  Found 16 models

  LogisticRegression... F1=0.9934 [0.9925, 0.9941]

  DecisionTree... F1=0.9991 [0.9987, 0.9994]

  RandomForest... F1=0.9993 [0.9990, 0.99

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

BASE = '/content/drive/MyDrive/MLmodeling/XAI'
catboost_results = []
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except:
            continue
    if not scores:
        return np.nan, np.nan, np.nan
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)
for ds_name, split_dir, model_path in [
    ('UNSW-NB15', f'{BASE}/splits/UNSW-NB15', f'{BASE}/models/UNSW-NB15/UNSW_09_CatBoost.pkl'),
    ('RT-IoT2022', f'{BASE}/splits/RT-IoT2022', f'{BASE}/models/RT-IoT2022/RT_09_CatBoost.pkl'),
    ('TON_IoT', f'{BASE}/splits/TON_IoT', f'{BASE}/models/TON_IoT/TON_09_CatBoost.pkl'),
    ('CICIoT2023', f'{BASE}/splits/CICIoT2023', f'{BASE}/models/CICIoT2023/CIC_09_CatBoost.pkl'),
]:
    print(f"\n{ds_name}...", end=" ")
    try:
        X_test_df = pd.read_csv(f'{split_dir}/X_test.csv')
        y_test = pd.read_csv(f'{split_dir}/y_test.csv').squeeze().astype(int)

        # Apply preprocessor if string columns exist
        if X_test_df.select_dtypes(include=['object','category']).shape[1] > 0:
            for ext in ['.pkl', '.joblib']:
                pp = f'{split_dir}/preprocessor{ext}'
                if os.path.exists(pp):
                    preprocessor = joblib.load(pp)
                    X_test = preprocessor.transform(X_test_df)
                    if hasattr(X_test, 'toarray'):
                        X_test = X_test.toarray()
                    X_test_df = pd.DataFrame(X_test)
                    break

        model = joblib.load(model_path)
        y_pred = model.predict(X_test_df)

        f1 = f1_score(y_test, y_pred, zero_division=0)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)

        f1_mean, f1_lo, f1_hi = bootstrap_ci(y_test, np.asarray(y_pred).flatten(), lambda yt,yp: f1_score(yt,yp,zero_division=0))
        prec_mean, prec_lo, prec_hi = bootstrap_ci(y_test, np.asarray(y_pred).flatten(), lambda yt,yp: precision_score(yt,yp,zero_division=0))
        rec_mean, rec_lo, rec_hi = bootstrap_ci(y_test, np.asarray(y_pred).flatten(), lambda yt,yp: recall_score(yt,yp,zero_division=0))

        print(f"F1={f1:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]")
        catboost_results.append({
            'dataset': ds_name, 'model': 'CatBoost',
            'test_f1': f"{f1:.4f}", 'f1_ci_lower': f"{f1_lo:.4f}", 'f1_ci_upper': f"{f1_hi:.4f}",
            'test_precision': f"{prec:.4f}", 'prec_ci_lower': f"{prec_lo:.4f}", 'prec_ci_upper': f"{prec_hi:.4f}",
            'test_recall': f"{rec:.4f}", 'rec_ci_lower': f"{rec_lo:.4f}", 'rec_ci_upper': f"{rec_hi:.4f}",
        })
    except Exception as e:
        print(f"FAILED: {e}")

cb_df = pd.DataFrame(catboost_results)
print("\n\nCatBoost Results:")
print(cb_df.to_string(index=False))

# Append to existing results
existing = pd.read_csv(f'{BASE}/bootstrap_results.csv')
existing = existing[~((existing['model']=='CatBoost') & (existing['test_f1']=='ERROR'))]
combined = pd.concat([existing, cb_df], ignore_index=True)
combined.to_csv(f'{BASE}/bootstrap_results.csv', index=False)
print(f"\nUpdated bootstrap_results.csv ({len(combined)} rows)")


UNSW-NB15... F1=0.9611 [0.9593, 0.9628]

RT-IoT2022... F1=0.9992 [0.9989, 0.9995]

TON_IoT... F1=0.9985 [0.9981, 0.9988]

CICIoT2023... F1=0.9980 [0.9979, 0.9980]


CatBoost Results:
   dataset    model test_f1 f1_ci_lower f1_ci_upper test_precision prec_ci_lower prec_ci_upper test_recall rec_ci_lower rec_ci_upper
 UNSW-NB15 CatBoost  0.9611      0.9593      0.9628         0.9650        0.9626        0.9673      0.9572       0.9548       0.9597
RT-IoT2022 CatBoost  0.9992      0.9989      0.9995         0.9991        0.9986        0.9995      0.9993       0.9989       0.9997
   TON_IoT CatBoost  0.9985      0.9981      0.9988         0.9989        0.9985        0.9993      0.9981       0.9975       0.9986
CICIoT2023 CatBoost  0.9980      0.9979      0.9980         0.9983        0.9983        0.9984      0.9976       0.9975       0.9977

Updated bootstrap_results.csv (55 rows)


In [ ]:
"""
==========================================================================
STATISTICAL SIGNIFICANCE TESTING
==========================================================================
Doctoral Praxis — Intrusion Detection with Explainable AI

Reads your CV fold data for all models × all datasets and produces:
  1. Friedman test per dataset (are any models significantly different?)
  2. Wilcoxon signed-rank pairwise tests (which pairs differ?)
  3. Summary tables ready to paste into your Chapter 4

INSTRUCTIONS:
  - Paste your CV data into the DATA section below (already done for you)
  - Run the whole file
  - Copy the output tables into your document
==========================================================================
"""

import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# ==========================================================================
# DATA — All 4 datasets, all models, 5-fold val_f1
# ==========================================================================

raw_data = []

# --- UNSW-NB15 ---
unsw = {
    "LogisticRegression": [0.9272863568, 0.9265830148, 0.9264556305, 0.9267012435, 0.925789616],
    "DecisionTree":       [0.9480618301, 0.9517181572, 0.9478662704, 0.9499663256, 0.9462085719],
    "RandomForest":       [0.9587133001, 0.9600398734, 0.9576007824, 0.9595983744, 0.9588583574],
    "ExtraTrees":         [0.9573888998, 0.9588892505, 0.9567354752, 0.957564415, 0.9570091022],
    "GradientBoosting":   [0.9476030733, 0.9488366086, 0.9471726673, 0.9480006939, 0.9484829534],
    "XGBoost":            [0.957808771, 0.9598294874, 0.9571590612, 0.9592611999, 0.9579374074],
    "AdaBoost":           [0.9305404309, 0.9356782842, 0.9334895611, 0.9321657411, 0.9345313405],
    "LightGBM":           [0.9543772039, 0.9573077761, 0.9549730449, 0.9559379852, 0.9555914589],
    "NaiveBayes":         [0.3646487215, 0.3739642467, 0.3684806454, 0.368596094, 0.366573382],
    "CatBoost":           [0.9589723424, 0.9590350973, 0.9577023499, 0.9597818975, 0.9596049683],
    "LinearSVM":          [0.927601335, 0.9272426709, 0.9264545835, 0.9274078428, 0.9266185844],
    "MLP":                [0.9498623071, 0.952278636, 0.950408743, 0.9508040809, 0.9501074137],
    "BaggingClassifier":  [0.9597596028, 0.9619181946, 0.9596607589, 0.9609242418, 0.9591052232],
    "KNN":                [0.9397758798, 0.9413042062, 0.9403644756, 0.940726828, 0.9409409409],
}
for model, folds in unsw.items():
    for i, f1 in enumerate(folds):
        raw_data.append({"dataset": "UNSW-NB15", "model": model, "fold": i+1, "val_f1": f1})

# --- TON-IoT ---
ton = {
    "LogisticRegression": [0.9722449884, 0.9732949513, 0.9707996805, 0.9712550697, 0.9729203109],
    "DecisionTree":       [0.9991127673, 0.9990244335, 0.9989355096, 0.9984694217, 0.9992238263],
    "RandomForest":       [0.9993789922, 0.9991354659, 0.9991575027, 0.9989580793, 0.9994456394],
    "ExtraTrees":         [0.9993790748, 0.999113318, 0.9989800443, 0.9989135496, 0.9992683414],
    "GradientBoosting":   [0.9969616997, 0.9970282977, 0.9964946532, 0.9968513016, 0.9970281659],
    "XGBoost":            [0.9993345459, 0.9989801348, 0.9990908889, 0.9991130427, 0.999334605],
    "AdaBoost":           [0.9927803413, 0.9929498088, 0.9930805792, 0.9921423211, 0.9926691397],
    "LightGBM":           [0.9992901823, 0.99920188, 0.9992018092, 0.999024347, 0.9993346345],
    "NaiveBayes":         [0.9969181651, 0.996811055, 0.9965885428, 0.9962321025, 0.99678371],
    "CatBoost":           [0.9987800821, 0.9985805536, 0.9982248258, 0.9984474116, 0.9984915705],
    "LinearSVM":          [0.9718160064, 0.9728412565, 0.9707656818, 0.9712968504, 0.9727743592],
    "MLP":                [0.9973379476, 0.9972947803, 0.9972052169, 0.9970730409, 0.9971617367],
    "KNN":                [0.9978930326, 0.99802079, 0.9979653048, 0.9979985918, 0.9979098636],
}
for model, folds in ton.items():
    for i, f1 in enumerate(folds):
        raw_data.append({"dataset": "TON-IoT", "model": model, "fold": i+1, "val_f1": f1})

# --- RT-IoT2022 ---
rt = {
    "LogisticRegression": [0.9926251779, 0.9932192444, 0.9937306101, 0.9931101407, 0.9928834832],
    "DecisionTree":       [0.9989670088, 0.9990957825, 0.9991605863, 0.9989991283, 0.9992249564],
    "RandomForest":       [0.9992251065, 0.9993865625, 0.9993220339, 0.9991607489, 0.9992572739],
    "ExtraTrees":         [0.9992573698, 0.9993865625, 0.9992575136, 0.9992897269, 0.9992895893],
    "GradientBoosting":   [0.9989019507, 0.998966742, 0.9991927933, 0.9987730854, 0.998966742],
    "XGBoost":            [0.9990637006, 0.9994833376, 0.9993866021, 0.9991605863, 0.9993865229],
    "AdaBoost":           [0.9968362603, 0.9974178555, 0.9974835463, 0.9969338024, 0.9977728285],
    "LightGBM":           [0.9991928454, 0.9994510285, 0.9994188674, 0.9990313206, 0.9992574657],
    "NaiveBayes":         [0.9689703808, 0.9694482737, 0.9704741448, 0.971070683, 0.9695569884],
    "CatBoost":           [0.9990637611, 0.9992895893, 0.9993220339, 0.9989992575, 0.9991926368],
    "LinearSVM":          [0.9919844861, 0.9919927677, 0.9932565418, 0.9923086867, 0.9916270649],
    "MLP":                [0.9988050254, 0.9987407575, 0.9986123213, 0.9985478718, 0.9983852215],
    "BaggingClassifier":  [0.9989024469, 0.9991603694, 0.9992897727, 0.9990313832, 0.9991927411],
    "KNN":                [0.9979671518, 0.9984834307, 0.9983221477, 0.9979341511, 0.9983210642],
}
for model, folds in rt.items():
    for i, f1 in enumerate(folds):
        raw_data.append({"dataset": "RT-IoT2022", "model": model, "fold": i+1, "val_f1": f1})

# --- CICIoT2023 ---
cic = {
    "LogisticRegression": [0.9925725057, 0.9924582861, 0.9925664832, 0.9927274802, 0.9926563693],
    "DecisionTree":       [0.9977035791, 0.9976334323, 0.9976253049, 0.9976675241, 0.9976681118],
    "RandomForest":       [0.9983549403, 0.9983654355, 0.9983095631, 0.9983667166, 0.9983337827],
    "ExtraTrees":         [0.9980216131, 0.9980047486, 0.998007197, 0.9980644035, 0.9980109204],
    "GradientBoosting":   [0.9976809444, 0.9976600469, 0.997647743, 0.9977467473, 0.9976999811],
    "XGBoost":            [0.9973521602, 0.9970841279, 0.9973233791, 0.9975737189, 0.9971741267],
    "AdaBoost":           [0.9962040724, 0.9959649793, 0.996009055, 0.9959724951, 0.9960123348],
    "LightGBM":           [0.9976785857, 0.9978621085, 0.9978831546, 0.9979129894, 0.9977718321],
    "NaiveBayes":         [0.9429074227, 0.9659343447, 0.9469754085, 0.9626573069, 0.9620246964],
    "CatBoost":           [0.9975311618, 0.9974142998, 0.9975384455, 0.9975526424, 0.9975079604],
    "LinearSVM":          [0.992602709, 0.9924503798, 0.9925655123, 0.9926899814, 0.9926274319],
    "MLP":                [0.9959151848, 0.9960177016, 0.9961707117, 0.9962117424, 0.9961131274],
    "BaggingClassifier":  [0.9981519156, 0.998095676, 0.9982074789, 0.9981981646, 0.9981546262],
    "KNN":                [0.9927698828, 0.9904001995, 0.9922751059, 0.9926461423, 0.991774676],
}
for model, folds in cic.items():
    for i, f1 in enumerate(folds):
        raw_data.append({"dataset": "CICIoT2023", "model": model, "fold": i+1, "val_f1": f1})

df = pd.DataFrame(raw_data)

# ==========================================================================
# TEST 1: FRIEDMAN TEST (per dataset)
# ==========================================================================

print("=" * 70)
print("  TEST 1: FRIEDMAN TEST — Are any models significantly different?")
print("=" * 70)
print()

friedman_results = []

for dataset in df["dataset"].unique():
    ds_data = df[df["dataset"] == dataset]
    models = sorted(ds_data["model"].unique())

    # Build matrix: rows = folds, columns = models
    fold_matrix = []
    for model in models:
        folds = ds_data[ds_data["model"] == model].sort_values("fold")["val_f1"].values
        fold_matrix.append(folds)

    fold_matrix = np.array(fold_matrix).T  # shape: (5 folds, N models)

    stat, p_value = stats.friedmanchisquare(*[fold_matrix[:, i] for i in range(fold_matrix.shape[1])])

    sig = "YES" if p_value < 0.05 else "NO"
    friedman_results.append({
        "Dataset": dataset,
        "Chi-sq": f"{stat:.4f}",
        "p-value": f"{p_value:.6f}",
        "Significant (p<0.05)": sig
    })

    print(f"{dataset}:")
    print(f"  Friedman chi-sq = {stat:.4f}, p = {p_value:.6f} → {sig}")
    print()

friedman_df = pd.DataFrame(friedman_results)
print("\nFriedman Summary Table:")
print(friedman_df.to_string(index=False))


# ==========================================================================
# TEST 2: WILCOXON SIGNED-RANK (pairwise, top models only)
# ==========================================================================

print("\n\n" + "=" * 70)
print("  TEST 2: WILCOXON SIGNED-RANK — Pairwise comparisons")
print("=" * 70)
print()
print("Comparing top-performing models + key architectural comparisons.")
print("Bonferroni correction applied.\n")

wilcoxon_results = []

for dataset in df["dataset"].unique():
    ds_data = df[df["dataset"] == dataset]
    models = sorted(ds_data["model"].unique())

    # Get mean F1 per model for ranking
    model_means = {}
    model_folds = {}
    for model in models:
        folds = ds_data[ds_data["model"] == model].sort_values("fold")["val_f1"].values
        model_means[model] = np.mean(folds)
        model_folds[model] = folds

    # Rank by mean F1
    ranked = sorted(model_means.items(), key=lambda x: x[1], reverse=True)

    # Compare: top 1 vs top 2, top 1 vs top 3, top 2 vs top 3,
    # best ensemble vs best baseline, best ensemble vs best neural
    pairs_to_test = []

    # Top 3 pairwise
    top3 = [r[0] for r in ranked[:3]]
    for a, b in combinations(top3, 2):
        pairs_to_test.append((a, b))

    # Best vs worst (sanity check)
    pairs_to_test.append((ranked[0][0], ranked[-1][0]))

    # Best ensemble vs best linear
    ensembles = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM",
                 "CatBoost", "GradientBoosting", "AdaBoost", "BaggingClassifier"]
    linears = ["LogisticRegression", "LinearSVM"]

    best_ens = None
    for r in ranked:
        if r[0] in ensembles:
            best_ens = r[0]
            break
    best_lin = None
    for r in ranked:
        if r[0] in linears:
            best_lin = r[0]
            break

    if best_ens and best_lin and (best_ens, best_lin) not in pairs_to_test:
        pairs_to_test.append((best_ens, best_lin))

    # Remove duplicates
    seen = set()
    unique_pairs = []
    for a, b in pairs_to_test:
        key = tuple(sorted([a, b]))
        if key not in seen:
            seen.add(key)
            unique_pairs.append((a, b))

    n_comparisons = len(unique_pairs)
    bonferroni_alpha = 0.05 / max(n_comparisons, 1)

    print(f"--- {dataset} ({n_comparisons} comparisons, Bonferroni α = {bonferroni_alpha:.4f}) ---")

    for model_a, model_b in unique_pairs:
        folds_a = model_folds[model_a]
        folds_b = model_folds[model_b]

        # Wilcoxon needs non-zero differences
        diffs = folds_a - folds_b
        if np.all(diffs == 0):
            print(f"  {model_a} vs {model_b}: identical — no test needed")
            wilcoxon_results.append({
                "Dataset": dataset,
                "Model A": model_a,
                "Model B": model_b,
                "Mean A": f"{np.mean(folds_a):.6f}",
                "Mean B": f"{np.mean(folds_b):.6f}",
                "Diff": f"{np.mean(diffs):.6f}",
                "p-value": "N/A",
                "Significant": "IDENTICAL"
            })
            continue

        try:
            stat, p_value = stats.wilcoxon(folds_a, folds_b, alternative="two-sided")
            sig = "YES" if p_value < bonferroni_alpha else "NO"
        except ValueError:
            # Can happen with too few non-zero differences
            p_value = 1.0
            sig = "NO (insufficient data)"

        winner = model_a if np.mean(folds_a) > np.mean(folds_b) else model_b

        print(f"  {model_a} vs {model_b}:")
        print(f"    Mean F1: {np.mean(folds_a):.6f} vs {np.mean(folds_b):.6f} (diff: {np.mean(diffs):+.6f})")
        print(f"    p = {p_value:.6f} → {'SIGNIFICANT' if sig == 'YES' else 'NOT significant'}")

        wilcoxon_results.append({
            "Dataset": dataset,
            "Model A": model_a,
            "Model B": model_b,
            "Mean A": f"{np.mean(folds_a):.6f}",
            "Mean B": f"{np.mean(folds_b):.6f}",
            "Diff": f"{np.mean(diffs):.6f}",
            "p-value": f"{p_value:.6f}",
            "Significant": sig
        })

    print()

wilcoxon_df = pd.DataFrame(wilcoxon_results)
print("\nWilcoxon Pairwise Summary Table:")
print(wilcoxon_df.to_string(index=False))


# ==========================================================================
# TEST 3: MODEL RANKING SUMMARY PER DATASET
# ==========================================================================

print("\n\n" + "=" * 70)
print("  MODEL RANKINGS BY DATASET (Mean CV F1)")
print("=" * 70)

for dataset in df["dataset"].unique():
    ds_data = df[df["dataset"] == dataset]
    models = ds_data["model"].unique()

    ranking = []
    for model in models:
        folds = ds_data[ds_data["model"] == model]["val_f1"].values
        ranking.append({
            "Model": model,
            "Mean F1": np.mean(folds),
            "Std F1": np.std(folds),
            "Min F1": np.min(folds),
            "Max F1": np.max(folds),
        })

    rank_df = pd.DataFrame(ranking).sort_values("Mean F1", ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    rank_df.index.name = "Rank"

    print(f"\n--- {dataset} ---")
    print(rank_df.to_string(
        formatters={
            "Mean F1": "{:.6f}".format,
            "Std F1": "{:.6f}".format,
            "Min F1": "{:.6f}".format,
            "Max F1": "{:.6f}".format,
        }
    ))


# ==========================================================================
# TEST 4: CROSS-DATASET CONSISTENCY
# ==========================================================================

print("\n\n" + "=" * 70)
print("  CROSS-DATASET: AVERAGE RANK PER MODEL")
print("=" * 70)

all_ranks = []

for dataset in df["dataset"].unique():
    ds_data = df[df["dataset"] == dataset]
    models = ds_data["model"].unique()

    means = {}
    for model in models:
        folds = ds_data[ds_data["model"] == model]["val_f1"].values
        means[model] = np.mean(folds)

    ranked = sorted(means.items(), key=lambda x: x[1], reverse=True)
    for rank, (model, mean_f1) in enumerate(ranked, 1):
        all_ranks.append({"Dataset": dataset, "Model": model, "Rank": rank})

ranks_df = pd.DataFrame(all_ranks)
avg_rank = ranks_df.groupby("Model")["Rank"].agg(["mean", "std", "min", "max"])
avg_rank = avg_rank.sort_values("mean").reset_index()
avg_rank.columns = ["Model", "Avg Rank", "Std Rank", "Best Rank", "Worst Rank"]

print("\nModels ranked by average position across all 4 datasets:")
print(avg_rank.to_string(index=False, formatters={
    "Avg Rank": "{:.2f}".format,
    "Std Rank": "{:.2f}".format,
    "Best Rank": "{:.0f}".format,
    "Worst Rank": "{:.0f}".format,
}))


print("\n\n" + "=" * 70)
print("  DONE — Copy tables above into Chapter 4")
print("=" * 70)

  TEST 1: FRIEDMAN TEST — Are any models significantly different?

UNSW-NB15:
  Friedman chi-sq = 64.4057, p = 0.000000 → YES

TON-IoT:
  Friedman chi-sq = 58.2330, p = 0.000000 → YES

RT-IoT2022:
  Friedman chi-sq = 61.5284, p = 0.000000 → YES

CICIoT2023:
  Friedman chi-sq = 64.0857, p = 0.000000 → YES


Friedman Summary Table:
   Dataset  Chi-sq  p-value Significant (p<0.05)
 UNSW-NB15 64.4057 0.000000                  YES
   TON-IoT 58.2330 0.000000                  YES
RT-IoT2022 61.5284 0.000000                  YES
CICIoT2023 64.0857 0.000000                  YES


  TEST 2: WILCOXON SIGNED-RANK — Pairwise comparisons

Comparing top-performing models + key architectural comparisons.
Bonferroni correction applied.

--- UNSW-NB15 (5 comparisons, Bonferroni α = 0.0100) ---
  BaggingClassifier vs CatBoost:
    Mean F1: 0.960274 vs 0.959019 (diff: +0.001254)
    p = 0.125000 → NOT significant
  BaggingClassifier vs RandomForest:
    Mean F1: 0.960274 vs 0.958962 (diff: +0.001311)
   

In [ ]:
## Per class F1

In [ ]:
import os, numpy as np, pandas as pd, joblib, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score

BASE = '/content/drive/MyDrive/MLmodeling/XAI'
DATASETS = {
    'UNSW-NB15': {'split_dir': f'{BASE}/splits/UNSW-NB15', 'model_dir': f'{BASE}/models/UNSW-NB15', 'prefix': 'UNSW'},
    'RT-IoT2022': {'split_dir': f'{BASE}/splits/RT-IoT2022', 'model_dir': f'{BASE}/models/RT-IoT2022', 'prefix': 'RT'},
    'TON_IoT': {'split_dir': f'{BASE}/splits/TON_IoT', 'model_dir': f'{BASE}/models/TON_IoT', 'prefix': 'TON'},
    'CICIoT2023': {'split_dir': f'{BASE}/splits/CICIoT2023', 'model_dir': f'{BASE}/models/CICIoT2023', 'prefix': 'CIC'},
}
SKIP = {'IsolationForest', 'LOF'}

results = []
for ds_name, cfg in DATASETS.items():
    X_df = pd.read_csv(f"{cfg['split_dir']}/X_test.csv")
    y_test = pd.read_csv(f"{cfg['split_dir']}/y_test.csv").squeeze().values

    # Apply preprocessor if needed
    if X_df.select_dtypes(include=['object','category']).shape[1] > 0:
        for ext in ['.pkl', '.joblib']:
            pp = f"{cfg['split_dir']}/preprocessor{ext}"
            if os.path.exists(pp):
                prep = joblib.load(pp)
                X_test = prep.transform(X_df)
                if hasattr(X_test, 'toarray'): X_test = X_test.toarray()
                X_test = np.asarray(X_test, dtype=float)
                break
    else:
        X_test = np.asarray(X_df, dtype=float)

    for f in sorted(os.listdir(cfg['model_dir'])):
        if not f.endswith('.pkl') or not f.startswith(cfg['prefix']): continue
        name = '_'.join(f.replace('.pkl','').split('_')[2:])
        if name in SKIP: continue

        try:
            model = joblib.load(os.path.join(cfg['model_dir'], f))
            exp_n = getattr(model, 'n_features_in_', 0)
            X_in = X_test
            if exp_n > 0 and X_test.shape[1] != exp_n:
                X_in = X_test[:, :exp_n] if X_test.shape[1] > exp_n else X_test

            # CatBoost fix
            if exp_n == 0: X_in = X_df if X_df.shape[1] >= X_test.shape[1] else X_test

            y_pred = model.predict(X_in)
            f1_normal = f1_score(y_test, y_pred, pos_label=0, zero_division=0)
            f1_attack = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
            f1_overall = f1_score(y_test, y_pred, zero_division=0)

            results.append({'dataset': ds_name, 'model': name,
                          'F1_overall': round(f1_overall,4),
                          'F1_normal': round(f1_normal,4),
                          'F1_attack': round(f1_attack,4)})
            print(f"{ds_name:15s} {name:25s} F1={f1_overall:.4f}  Normal={f1_normal:.4f}  Attack={f1_attack:.4f}")
        except Exception as e:
            print(f"{ds_name:15s} {name:25s} FAILED: {e}")

df = pd.DataFrame(results)
df.to_csv(f'{BASE}/per_class_f1.csv', index=False)
print(f"\nSaved to {BASE}/per_class_f1.csv")

UNSW-NB15       LogisticRegression        F1=0.9286  Normal=0.8576  Attack=0.9286
UNSW-NB15       DecisionTree              F1=0.9521  Normal=0.9152  Attack=0.9521
UNSW-NB15       RandomForest              F1=0.9612  Normal=0.9316  Attack=0.9612
UNSW-NB15       ExtraTrees                F1=0.9607  Normal=0.9308  Attack=0.9607
UNSW-NB15       GradientBoosting          F1=0.9495  Normal=0.9103  Attack=0.9495
UNSW-NB15       AdaBoost                  F1=0.9315  Normal=0.8724  Attack=0.9315
UNSW-NB15       XGBoost                   F1=0.9591  Normal=0.9292  Attack=0.9591
UNSW-NB15       LightGBM                  F1=0.9575  Normal=0.9263  Attack=0.9575
UNSW-NB15       CatBoost                  F1=0.9611  Normal=0.9318  Attack=0.9611
UNSW-NB15       KNN                       F1=0.9442  Normal=0.9025  Attack=0.9442
UNSW-NB15       NaiveBayes                F1=0.3655  Normal=0.5926  Attack=0.3655
UNSW-NB15       LinearSVM                 F1=0.9291  Normal=0.8617  Attack=0.9291
UNSW-NB15       